# Missed demand estimation
Goal: estimate how much customers would have bought during empty-shelf hours. We test methods by hiding hours we already know (fully stocked days), filling them back in, and comparing with the real sales.

In [1]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')

token = userdata.get('GITHUB_TOKEN')
repo = 'stockout-aware-forecasting'
if not os.path.exists(f'/content/{repo}'):
    !git clone https://{token}@github.com/bulut19/{repo}.git /content/{repo}
%cd /content/{repo}
!git pull

Mounted at /content/drive
Cloning into '/content/stockout-aware-forecasting'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 51 (delta 15), reused 5 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 286.36 KiB | 5.61 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/stockout-aware-forecasting
Already up to date.


In [2]:
from src.load_data import DRIVE_DIR, load_daily, load_hourly
print(DRIVE_DIR)
print(load_daily().shape)

/content/drive/MyDrive/MIS 382 Deep Learning and ML/Final Project/Stockout Project Data
(4850000, 20)


In [3]:
import numpy as np, pandas as pd
from src.load_data import PROC_DIR, load_hourly
from src.metrics import recovery_error

sample_ids = pd.read_csv(PROC_DIR / 'sample_ids.csv').series_id.tolist()
hourly = load_hourly(sample_ids)
hourly = hourly[hourly.hour.between(6, 21)].reset_index(drop=True)   # selling hours only
hourly['weekend'] = (hourly.dt.dt.dayofweek >= 5).astype('int8')
hourly['day_oos_hours'] = hourly.groupby(['series_id', 'dt']).out_of_stock.transform('sum')

print(f'{len(hourly):,} hourly rows for {hourly.series_id.nunique():,} products')
print('Share of selling hours out of stock:', round(hourly.out_of_stock.mean(), 3))

3,104,000 hourly rows for 2,000 products
Share of selling hours out of stock: 0.199


## Masking test
On a random 10% of fully stocked training days, we hide one block of 2 to 6 hours. Block start times are drawn from real stockout start times, so the test looks like real stockouts.

In [4]:
rng = np.random.default_rng(42)
first_out = hourly[hourly.out_of_stock == 1].groupby(['series_id', 'dt']).hour.min()
start_hours = first_out[first_out >= 8].values

full_days = hourly.loc[(hourly.day_oos_hours == 0) & (hourly.split == 'train'), ['series_id', 'dt']].drop_duplicates()
test_days = full_days.sample(frac=0.10, random_state=42).reset_index(drop=True)
test_days['start'] = rng.choice(start_hours, len(test_days))
test_days['length'] = rng.integers(2, 7, len(test_days))
test_days['end'] = np.minimum(test_days.start + test_days.length, 22)

hourly = hourly.merge(test_days, on=['series_id', 'dt'], how='left')
hourly['masked'] = ((hourly.hour >= hourly.start) & (hourly.hour < hourly.end)).astype('int8')
hourly = hourly.drop(columns=['start', 'length', 'end'])
hourly['observed'] = (hourly.out_of_stock == 0) & (hourly.masked == 0)   # hours a method is allowed to learn from

print(f'Test days: {len(test_days):,} | hidden hours: {hourly.masked.sum():,} | avg block {(test_days.end - test_days.start).mean():.1f} h')

Test days: 10,008 | hidden hours: 35,373 | avg block 3.5 h


## Simple methods
- **Historical average:** the product's normal sales at that hour (weekday vs. weekend), from in-stock training hours.
- **Day-adjusted average:** the historical average scaled by how busy that day is in the hours we can see. Targets the finding that demand runs high right before stockouts.

In [5]:
def historical_average(df):
    """Normal sales for this product, hour and weekday/weekend, from observed training hours."""
    obs = df[df.observed & (df.split == 'train')]
    p1 = obs.groupby(['series_id', 'hour', 'weekend']).sales.mean().rename('p1')
    p2 = obs.groupby(['series_id', 'hour']).sales.mean().rename('p2')   # fallback
    out = df.join(p1, on=['series_id', 'hour', 'weekend']).join(p2, on=['series_id', 'hour'])
    return out.p1.fillna(out.p2).fillna(0).to_numpy()

def day_adjusted_average(df, shrink=4.0):
    """Historical average scaled by how busy the day is in the hours we can see."""
    base_est = historical_average(df)
    tmp = df[['series_id', 'dt', 'sales', 'observed']].assign(base=base_est)
    seen = tmp[tmp.observed].groupby(['series_id', 'dt']).agg(actual=('sales', 'sum'), expected=('base', 'sum'))
    seen['factor'] = (seen.actual + shrink * 0.1) / (seen.expected + shrink * 0.1)   # pulls toward 1 when little is visible
    factor = tmp.join(seen.factor, on=['series_id', 'dt']).factor.fillna(1).clip(0.2, 5).to_numpy()
    return base_est * factor

hourly['est_avg'] = historical_average(hourly)
hourly['est_dayadj'] = day_adjusted_average(hourly)

# Score on total hidden demand per test day (hourly values are too small and noisy to score one by one)
blocks = hourly[hourly.masked == 1].groupby(['series_id', 'dt'])[['sales', 'est_avg', 'est_dayadj']].sum()
results = pd.DataFrame({
    'Raw sales (assume zero)': recovery_error(blocks.sales, np.zeros(len(blocks))),
    'Historical average': recovery_error(blocks.sales, blocks.est_avg),
    'Day-adjusted average': recovery_error(blocks.sales, blocks.est_dayadj),
}).T.astype(float).round(3)
results

,wape,bias
Raw sales (assume zero),1.000,-1.000
Historical average,0.678,0.135
Day-adjusted average,0.654,0.048


**Findings:**
- Treating empty hours as zero misses 100% of hidden demand (bias -1.0), which is what forecasting on raw sales does.
- The historical average cuts the error to 68% but over-estimates by 13.5% on this test. Fully stocked days are quieter than average, the mirror image of the 1.79x finding before stockouts.
- The day-adjusted average does best: 65% error and only +4.8% bias, because it adapts to how busy each day is.
- Error is still high because hidden blocks are small and noisy (about 0.19 units on average). Next: LightGBM and an LSTM to beat the day-adjusted baseline.

## LightGBM
LightGBM predicts hidden-hour sales from the product's normal pattern, how busy the visible part of the day is, nearby visible hours, discount, holiday and activity flags, weather and category. It trains on **practice blocks**: hours hidden on other fully stocked training days (never the test days), so it learns the same task it is tested on.

In [6]:
import lightgbm as lgb

# 1. Add daily context (discount, flags, weather, category)
daily_cols = ['series_id', 'dt', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'first_category_id']
hourly = hourly.merge(pd.read_parquet(PROC_DIR / 'daily_model.parquet', columns=daily_cols), on=['series_id', 'dt'], how='left')
hourly = hourly.sort_values(['series_id', 'dt', 'hour']).reset_index(drop=True)
hourly['day_of_week'] = hourly.dt.dt.dayofweek

# 2. Practice blocks: hide blocks on OTHER fully stocked training days
rng = np.random.default_rng(7)
day_tbl = hourly.groupby(['series_id', 'dt']).agg(oos=('day_oos_hours', 'first'), split=('split', 'first'), test=('masked', 'max')).reset_index()
pool = day_tbl[(day_tbl.oos == 0) & (day_tbl.split == 'train') & (day_tbl.test == 0)]
practice = pool.sample(frac=0.4, random_state=7)[['series_id', 'dt']].reset_index(drop=True)
practice['start'] = rng.choice(start_hours, len(practice))
practice['end'] = np.minimum(practice.start + rng.integers(2, 7, len(practice)), 22)
hourly = hourly.merge(practice, on=['series_id', 'dt'], how='left')
hourly['practice'] = ((hourly.hour >= hourly.start) & (hourly.hour < hourly.end)).astype('int8')
hourly = hourly.drop(columns=['start', 'end'])
visible = hourly.observed & (hourly.practice == 0)   # hours a method is allowed to see

# 3. Features built only from visible hours
keys = [hourly.series_id, hourly.dt]
vis_sales, vis_base = hourly.sales.where(visible, 0.0), hourly.est_avg.where(visible, 0.0)
hourly['day_factor'] = (vis_sales.groupby(keys).transform('sum') + 0.4) / (vis_base.groupby(keys).transform('sum') + 0.4)
hourly['day_vis_hours'] = visible.groupby(keys).transform('sum')
hourly['dayadj'] = hourly.est_avg * hourly.day_factor.clip(0.2, 5)
hourly['last_vis_sales'] = hourly.sales.where(visible).groupby(keys).shift(1).groupby(keys).ffill()
hourly['next_vis_sales'] = hourly.sales.where(visible).groupby(keys).shift(-1).groupby(keys).bfill()
vis_hour = hourly.hour.where(visible).groupby(keys).shift(1).groupby(keys).ffill()
hourly['hours_since_vis'] = hourly.hour - vis_hour

FEATURES = ['hour', 'weekend', 'day_of_week', 'est_avg', 'dayadj', 'day_factor', 'day_vis_hours', 'last_vis_sales',
            'next_vis_sales', 'hours_since_vis', 'discount', 'holiday_flag', 'activity_flag', 'precpt',
            'avg_temperature', 'first_category_id']

# 4. Train on practice blocks, then fill the test blocks
train_rows = hourly.practice == 1
model_lgb = lgb.LGBMRegressor(objective='tweedie', n_estimators=500, learning_rate=0.03, num_leaves=31,
                              min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                              random_state=42, verbose=-1)
model_lgb.fit(hourly.loc[train_rows, FEATURES], hourly.loc[train_rows, 'sales'], categorical_feature=['first_category_id'])
hourly['est_lgb'] = model_lgb.predict(hourly[FEATURES])

blocks = hourly[hourly.masked == 1].groupby(['series_id', 'dt'])[['sales', 'est_avg', 'est_dayadj', 'est_lgb']].sum()
results = pd.DataFrame({
    'Raw sales (assume zero)': recovery_error(blocks.sales, np.zeros(len(blocks))),
    'Historical average': recovery_error(blocks.sales, blocks.est_avg),
    'Day-adjusted average': recovery_error(blocks.sales, blocks.est_dayadj),
    'LightGBM': recovery_error(blocks.sales, blocks.est_lgb),
}).T.astype(float).round(3)
print(f'Practice hours used for training: {train_rows.sum():,}')
display(results)
imp = pd.Series(model_lgb.booster_.feature_importance('gain'), index=FEATURES)
(imp / imp.sum()).sort_values(ascending=False).head(8).round(3)

Practice hours used for training: 128,048


,wape,bias
Raw sales (assume zero),1.000,-1.000
Historical average,0.678,0.135
Day-adjusted average,0.654,0.048
LightGBM,0.622,-0.002


,0
est_avg,0.515
dayadj,0.304
first_category_id,0.034
discount,0.028
day_factor,0.025
avg_temperature,0.023
precpt,0.021
last_vis_sales,0.014


**Findings:**
- LightGBM is the best so far: 62% error (vs. 65% for the day-adjusted average) and essentially zero bias (-0.2%).
- Training on practice blocks mattered: the model learns from situations that look like real stockouts, not from normal hours.
- The product's normal pattern and how busy the day is carry most of the signal; discount, weather and category add a little.
- Caveat: test blocks are on fully stocked days, which are quieter than real stockout days, so real missed demand is likely somewhat higher than these estimates.

## LSTM (deep learning)
A two-way LSTM reads each product-day as a sequence of 16 selling hours, forwards and backwards, so it uses both the hours before and after a gap. Each hour's input: visible sales (0 if hidden), a visible/hidden flag, the product's normal pattern, the hour, and the day's discount, flags and weather, plus a learned category embedding.

It trains on all fully stocked training days (never the test days). Every epoch hides fresh random blocks, and the loss only counts hidden hours, so it learns to fill gaps.

In [7]:
import torch, torch.nn as nn
torch.manual_seed(42); np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Reshape into one sequence of 16 selling hours per product-day
hourly = hourly.sort_values(['series_id', 'dt', 'hour']).reset_index(drop=True)
n_days = len(hourly) // 16
def as_days(col):
    return hourly[col].to_numpy(dtype='float32').reshape(n_days, 16)

sales, base, observed, masked = as_days('sales'), as_days('est_avg'), as_days('observed'), as_days('masked')
scale = np.maximum(hourly.groupby('series_id').est_avg.transform('mean').to_numpy('float32').reshape(n_days, 16)[:, :1], 0.01)
day_info = hourly.groupby(['series_id', 'dt'], sort=True).agg(
    discount=('discount', 'first'), holiday=('holiday_flag', 'first'), activity=('activity_flag', 'first'),
    weekend=('weekend', 'first'), rain=('precpt', 'first'), temp=('avg_temperature', 'first'),
    category=('first_category_id', 'first'), oos=('day_oos_hours', 'first'), split=('split', 'first')).reset_index()
day_feats = np.stack([day_info.discount.fillna(1).values, day_info.holiday.values, day_info.activity.values,
                      day_info.weekend.values, day_info.rain.values / 10, (day_info.temp.values - 22) / 4], axis=1).astype('float32')
category = day_info.category.astype('category').cat.codes.to_numpy('int64')
hour_pos = np.tile(np.linspace(0, 1, 16, dtype='float32'), (n_days, 1))

is_test_day = masked.max(axis=1) == 1
train_pool = np.where((day_info.oos.values == 0) & (day_info.split.values == 'train') & ~is_test_day)[0]
test_idx = np.where(is_test_day)[0]
start_choices = np.clip(start_hours - 6, 0, 15)   # real stockout start hours, as positions 0-15

def make_inputs(idx, visible):
    """Per-hour inputs: visible sales, visibility flag, normal pattern, hour, plus day context."""
    s = sales[idx] / scale[idx]; b = base[idx] / scale[idx]
    x = np.stack([s * visible, visible, b, hour_pos[idx]], axis=2)
    d = np.repeat(day_feats[idx][:, None, :], 16, axis=1)
    return torch.tensor(np.concatenate([x, d], axis=2)), torch.tensor(category[idx])

def random_blocks(n):
    """Hide one block of 2-6 hours per day, starting at a real stockout start time."""
    start = np.random.choice(start_choices, n); end = np.minimum(start + np.random.randint(2, 7, n), 16)
    pos = np.arange(16)[None, :]
    return ((pos >= start[:, None]) & (pos < end[:, None])).astype('float32')

# 2. The model: a two-way LSTM reads the day's hours forwards and backwards
class Filler(nn.Module):
    def __init__(self, n_in, n_cat, hidden=64):
        super().__init__()
        self.cat_emb = nn.Embedding(n_cat, 4)
        self.lstm = nn.LSTM(n_in + 4, hidden, batch_first=True, bidirectional=True, num_layers=2, dropout=0.1)
        self.head = nn.Sequential(nn.Linear(2 * hidden, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x, cat):
        c = self.cat_emb(cat)[:, None, :].expand(-1, x.shape[1], -1)
        out, _ = self.lstm(torch.cat([x, c], dim=2))
        return nn.functional.softplus(self.head(out).squeeze(-1))

n_in = 4 + day_feats.shape[1]
model = Filler(n_in, category.max() + 1).to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

# 3. Train: every epoch hides fresh random blocks; the loss only counts hidden hours
EPOCHS, BATCH = 8, 512
for epoch in range(EPOCHS):
    model.train(); np.random.shuffle(train_pool); total = 0
    for i in range(0, len(train_pool), BATCH):
        idx = train_pool[i:i + BATCH]
        hide = random_blocks(len(idx))
        x, cat = make_inputs(idx, 1 - hide)
        target = torch.tensor(sales[idx] / scale[idx]); hide_t = torch.tensor(hide)
        pred = model(x.to(device), cat.to(device))
        loss = (((pred - target.to(device)) ** 2) * hide_t.to(device)).sum() / hide_t.sum()
        opt.zero_grad(); loss.backward(); opt.step(); total += loss.item() * len(idx)
    print(f'epoch {epoch + 1}: training loss {total / len(train_pool):.4f}')

# 4. Fill the test blocks
model.eval()
with torch.no_grad():
    x, cat = make_inputs(test_idx, observed[test_idx])
    pred = model(x.to(device), cat.to(device)).cpu().numpy() * scale[test_idx]
est_lstm = np.zeros_like(sales); est_lstm[test_idx] = pred
hourly['est_lstm'] = est_lstm.reshape(-1)

blocks = hourly[hourly.masked == 1].groupby(['series_id', 'dt'])[['sales', 'est_avg', 'est_dayadj', 'est_lgb', 'est_lstm']].sum()
results = pd.DataFrame({
    'Raw sales (assume zero)': recovery_error(blocks.sales, np.zeros(len(blocks))),
    'Historical average': recovery_error(blocks.sales, blocks.est_avg),
    'Day-adjusted average': recovery_error(blocks.sales, blocks.est_dayadj),
    'LightGBM': recovery_error(blocks.sales, blocks.est_lgb),
    'LSTM': recovery_error(blocks.sales, blocks.est_lstm),
    'Average of LightGBM + LSTM': recovery_error(blocks.sales, (blocks.est_lgb + blocks.est_lstm) / 2),
}).T.astype(float).round(3)
results

epoch 1: training loss 2.3813
epoch 2: training loss 2.3717
epoch 3: training loss 2.3292
epoch 4: training loss 2.3218
epoch 5: training loss 2.3112
epoch 6: training loss 2.3006
epoch 7: training loss 2.2963
epoch 8: training loss 2.2761


,wape,bias
Raw sales (assume zero),1.000,-1.000
Historical average,0.678,0.135
Day-adjusted average,0.654,0.048
LightGBM,0.622,-0.002
LSTM,0.625,-0.002
Average of LightGBM + LSTM,0.620,-0.002


**Findings:**
- The LSTM (62.5% error) matches LightGBM (62.2%), and both beat the simple methods by a clear margin. Both are essentially unbiased.
- Averaging the two gives the best result (62.0%), so we use the LightGBM + LSTM average to fill real empty-shelf hours.
- The models agree closely, which suggests we are near the limit of what this data can predict: hidden blocks are small (about 0.19 units), so a lot of hour-to-hour variation is random.
- Compared with treating empty hours as zero, this recovers the hidden demand with no systematic under- or over-estimate on the test.

## Corrected demand (v1)
Every real empty-shelf hour is filled with the average of the LightGBM and LSTM estimates. In-stock hours keep their actual sales, and an estimate is never allowed to fall below what actually sold. Saved for the team as `hourly_demand_v1.parquet` and `daily_demand_v1.parquet`.

In [8]:
# 1. Real filling: every in-stock hour is visible (test blocks were artificial), every empty-shelf hour gets filled
real_visible = hourly.out_of_stock == 0
keys = [hourly.series_id, hourly.dt]
vis_sales, vis_base = hourly.sales.where(real_visible, 0.0), hourly.est_avg.where(real_visible, 0.0)
feat = hourly.copy()
feat['day_factor'] = (vis_sales.groupby(keys).transform('sum') + 0.4) / (vis_base.groupby(keys).transform('sum') + 0.4)
feat['day_vis_hours'] = real_visible.groupby(keys).transform('sum')
feat['dayadj'] = feat.est_avg * feat.day_factor.clip(0.2, 5)
feat['last_vis_sales'] = hourly.sales.where(real_visible).groupby(keys).shift(1).groupby(keys).ffill()
feat['next_vis_sales'] = hourly.sales.where(real_visible).groupby(keys).shift(-1).groupby(keys).bfill()
feat['hours_since_vis'] = hourly.hour - hourly.hour.where(real_visible).groupby(keys).shift(1).groupby(keys).ffill()
fill_lgb = model_lgb.predict(feat[FEATURES])

all_idx = np.arange(n_days)
model.eval(); fill_lstm = np.zeros_like(sales)
with torch.no_grad():
    for i in range(0, n_days, 4096):
        idx = all_idx[i:i + 4096]
        x, cat = make_inputs(idx, 1 - as_days('out_of_stock')[idx])
        fill_lstm[idx] = model(x.to(device), cat.to(device)).cpu().numpy() * scale[idx]

hourly['demand_est'] = (fill_lgb + fill_lstm.reshape(-1)) / 2
# Corrected demand: real sales when in stock; the estimate (never below actual sales) when the shelf was empty
hourly['demand'] = np.where(hourly.out_of_stock == 1, np.maximum(hourly.sales, hourly.demand_est), hourly.sales)
hourly['lost_demand'] = hourly.demand - hourly.sales

# 2. Daily totals, with the frozen train / valid / test labels
daily_corr = hourly.groupby(['series_id', 'dt']).agg(sales=('sales', 'sum'), demand=('demand', 'sum'),
                                                       lost_demand=('lost_demand', 'sum'), oos_hours=('out_of_stock', 'sum')).reset_index()
daily_corr = daily_corr.merge(pd.read_parquet(PROC_DIR / 'daily_model.parquet', columns=['series_id', 'dt', 'set']),
                              on=['series_id', 'dt'], how='left')
daily_corr['zero_sales_day'] = (daily_corr.sales == 0).astype('int8')

# 3. Headline numbers
lost_share = daily_corr.lost_demand.sum() / daily_corr.sales.sum()
so_days = daily_corr[daily_corr.oos_hours > 0]
print(f'Missed demand = {lost_share:.1%} of recorded sales (selling hours, 2,000 products)')
print(f'On stockout days, true demand is {so_days.demand.sum() / so_days.sales.sum() - 1:.1%} above recorded sales')
print(f'Share of missed demand on days that still show sales (invisible in daily data): '
      f'{daily_corr.loc[daily_corr.zero_sales_day == 0, "lost_demand"].sum() / daily_corr.lost_demand.sum():.1%}')

# 4. Save for the team
cols = ['series_id', 'dt', 'hour', 'split', 'sales', 'out_of_stock', 'demand_est', 'demand', 'lost_demand']
hourly[cols].to_parquet(PROC_DIR / 'hourly_demand_v1.parquet', index=False)
daily_corr.to_parquet(PROC_DIR / 'daily_demand_v1.parquet', index=False)
print(f'Saved hourly_demand_v1.parquet ({len(hourly):,} rows) and daily_demand_v1.parquet ({len(daily_corr):,} rows)')

Missed demand = 20.7% of recorded sales (selling hours, 2,000 products)
On stockout days, true demand is 45.3% above recorded sales
Share of missed demand on days that still show sales (invisible in daily data): 80.9%
Saved hourly_demand_v1.parquet (3,104,000 rows) and daily_demand_v1.parquet (194,000 rows)


**Findings (corrected demand v1):**
- **Stockouts hide about 21% of demand:** true demand is roughly 20.7% higher than recorded sales across the 2,000 products.
- **On stockout days, true demand is about 45% above what was recorded.** A forecast trained on raw sales learns these days as much quieter than they were.
- **About 81% of missed demand happens on days that still show some sales**, so a daily system would never flag it. This is the core of the daily vs. hourly story.
- These are likely conservative: the methods were tested on quieter, fully stocked days, while real stockouts happen on busy days.

   ## Save to GitHub
   Press Ctrl+S first, then run the two cells below. Update the commit message each time.